Dataset

In [ ]:
!pip install lightning

from google.colab import drive
drive.mount('/content/drive')

!cp /content/drive/MyDrive/hw4_realse_dataset.zip /content/
!unzip -qo hw4_realse_dataset.zip

!git clone https://github.com/va1shn9v/PromptIR.git


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 82.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 95.2 MB/s eta 0:00:00
Mounted at /content/drive
Cloning into 'PromptIR'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 126 (delta 41), reused 34 (delta 34), pack-reused 78 (from 1)
Receiving objects: 100% (126/126), 1.37 MiB | 26.00 MiB/s, done.
Resolving deltas: 100% (53/53), done.


Data management

In [ ]:
import os
import shutil
import random

# Configuration
base_path = 'hw4_realse_dataset'
train_in = os.path.join(base_path, 'train/degraded')
train_out = os.path.join(base_path, 'train/clean')

tasks = ['rain', 'snow']

for task in tasks:
    # Create directories
    os.makedirs(f'/content/PromptIR/data/Train/Derain/rainy', exist_ok=True)
    os.makedirs(f'/content/PromptIR/data/Train/Derain/gt', exist_ok=True)
    os.makedirs(f'/content/PromptIR/data/Train/Dehaze/synthetic', exist_ok=True)
    os.makedirs(f'/content/PromptIR/data/Train/Dehaze/original', exist_ok=True)

    files = [f for f in os.listdir(train_in) if f.startswith(task)]
    i = 1601
    for f in files:
        src_degraded = os.path.join(train_in, f)
        # Match naming: rain-1.png -> rain_clean-1.png
        clean_name = f.replace(f'{task}-', f'{task}_clean-')
        src_clean = os.path.join(train_out, clean_name)

        # Destination paths
        if task == 'rain':
          dest_in = f'/content/PromptIR/data/Train/Derain/rainy/{f}'
          dest_out = f'/content/PromptIR/data/Train/Derain/gt/norain-{f.split('rain-')[-1]}'

        else:
          dest_in = f'/content/PromptIR/data/Train/Dehaze/synthetic/{f}'
          dest_out = f'/content/PromptIR/data/Train/Dehaze/original/nosnow-{f.split('snow-')[-1]}'


        if os.path.exists(src_clean):
            shutil.move(src_degraded, dest_in)
            shutil.move(src_clean, dest_out)


rainTxt = "/content/PromptIR/data_dir/rainy/rainTrain.txt"
snowTxt = "/content/PromptIR/data_dir/hazy/hazy_outside.txt"
with open(rainTxt, "w") as f:
    for i in range(1, 1601):
        f.write(f"rainy/rain-{i}.png\n")
with open(snowTxt, "w") as f:
    for i in range(1, 1601):
        f.write(f"synthetic/snow-{i}.png\n")

Training

In [ ]:
%cd PromptIR
!python train.py  --de_type derain dehaze\
          --batch_size 24 \
          --patch_size 128 \
          --lr 2e-4 \
          --epochs 200 \
          --num_gpus 1 \
          --num_workers 16 \
          --ckpt_dir 'ckpt'


Inference

In [ ]:
!python demo.py  --test_path '../hw4_realse_dataset/test/degraded/' \
          --output_path '../output/' \

%cd /content

Generate .npz file

In [ ]:
import os
import numpy as np
from PIL import Image

# Set your image folder path
folder_path = 'output'
output_npz = 'pred.npz'

# Initialize dictionary to hold image arrays
images_dict = {}

# Loop through all files in the folder
for filename in os.listdir(folder_path):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        file_path = os.path.join(folder_path, filename)
        # Load image and convert to RGB
        image = Image.open(file_path).convert('RGB')
        img_array = np.array(image)
        # Rearrange to (3, H, W)
        img_array = np.transpose(img_array, (2, 0, 1))
        # Add to dictionary
        images_dict[filename] = img_array

# Save to .npz file
np.savez(output_npz, **images_dict)
print(f"Saved {len(images_dict)} images to {output_npz}")
